# 19 Differential Abundance and Body Mass Index (BMI)

In the first part of this notebook, we use ANCOM-BC2 to test whether fungal genus abundance is associated with numeric BMI values (`bmi_sample`). The second section treats BMI as categorical labels (`bmi_group`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for age, sex, and diet, as in notebook 12.


## 1. Setup

### 1.1 Imports

In [1]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [2]:
metadata = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex_with_BMI_groups.tsv"
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/19_differential_abundance_BMI"
vis_dir = "../data/processed/19_differential_abundance_BMI"

genus_table = f"{in_dir}/table_fungi_l6.qza"
BMI_table = f"{out_dir}/table_fungi_l6_BMI.qza"

!mkdir -p "$out_dir" "$vis_dir"


# Numeric BMI
num_out_dir = f"{out_dir}/numeric_BMI"
num_vis_dir = f"{vis_dir}/numeric_BMI"

num_unadjusted_dir = f"{num_vis_dir}/unadjusted_num_ancombc2" 
num_adjusted_dir = f"{num_vis_dir}/adjusted_num_ancombc2"

num_unadjusted_qza = f"{num_out_dir}/ancombc2_BMI_num_unadjusted.qza"
num_adjusted_qza = f"{num_out_dir}/ancombc2_BMI_num_adjusted.qza"

num_unadjusted_tabulate = f"{num_unadjusted_dir}/tabulate_BMI_num_unadjusted.qzv"
num_adjusted_tabulate = f"{num_adjusted_dir}/tabulate_BMI_num_adjusted.qzv"

num_unadjusted_barplot = f"{num_unadjusted_dir}/barplot_BMI_num_unadjusted.qzv"
num_adjusted_barplot = f"{num_adjusted_dir}/barplot_BMI_num_adjusted.qzv"

!mkdir -p "$num_out_dir" "$num_vis_dir"
!mkdir -p "$num_unadjusted_dir" "$num_adjusted_dir"


# Categorical BMI
cat_out_dir = f"{out_dir}/categorical_BMI"
cat_vis_dir = f"{vis_dir}/categorical_BMI"

cat_unadjusted_dir = f"{cat_vis_dir}/unadjusted_cat_ancombc2" 
cat_adjusted_dir = f"{cat_vis_dir}/adjusted_cat_ancombc2"

cat_unadjusted_qza = f"{cat_out_dir}/ancombc2_BMI_cat_unadjusted.qza"
cat_adjusted_qza = f"{cat_out_dir}/ancombc2_BMI_cat_adjusted.qza"

cat_unadjusted_tabulate = f"{cat_unadjusted_dir}/tabulate_BMI_cat_unadjusted.qzv"
cat_adjusted_tabulate = f"{cat_adjusted_dir}/tabulate_BMI_cat_adjusted.qzv"

cat_unadjusted_barplot = f"{cat_unadjusted_dir}/barplot_BMI_cat_unadjusted.qzv"
cat_adjusted_barplot = f"{cat_adjusted_dir}/barplot_BMI_cat_adjusted.qzv"

!mkdir -p "$cat_out_dir" "$cat_vis_dir"
!mkdir -p "$cat_unadjusted_dir" "$cat_adjusted_dir"



### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 09.

In [5]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 149


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and diet information so both models use the same samples.

In [18]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[age_years_sample] IS NOT NULL AND [sex_sample] IS NOT NULL AND [diet_type_sample] IS NOT NULL AND [bmi_sample] IS NOT NULL AND [bmi_group] IS NOT NULL" \
    --o-filtered-table "$BMI_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/19_differential_abundance_BMI/table_fungi_l6_BMI.qza


### 2.2 Summarize selected samples

We summarize the numeric age values after filtering for complete settlement, sex, and age information.

In [6]:
BMI_counts = Artifact.load(BMI_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[BMI_counts.index]

print("Selected samples:", len(selected_metadata))
display(selected_metadata["bmi_sample"].describe())
display(selected_metadata["bmi_group"].value_counts())

Selected samples: 139


count    139.000000
mean      24.251727
std        6.097824
min       14.350000
25%       21.015000
50%       23.310000
75%       26.810000
max       71.610000
Name: bmi_sample, dtype: float64

bmi_group
Healthy weight    84
Overweight        33
Obese             12
Underweight       10
Name: count, dtype: int64

## 3. BMI as a continuous predictor

### 3.1 Unadjusted ANCOM-BC2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

We test whether fungal genus abundance is associated with continuous BMI. For each genus, the log fold change tells us how much its predicted log abundance differs per 1 kg/m² increase in BMI.

In [4]:
# check if the values inside the metadata table are registered as numeric.
print(Metadata.load(metadata).get_column("bmi_sample").type) 

numeric


In [16]:
!qiime composition ancombc2 \
    --i-table "$BMI_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "bmi_sample" \
    --p-diff-robust \
    --o-ancombc2-output "$num_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/19_differential_abundance_BMI/numeric_BMI/ancombc2_BMI_num_unadjusted.qza


In [12]:
!qiime composition tabulate \
    --i-data "$num_unadjusted_qza" \
    --o-visualization "$num_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$num_unadjusted_qza" \
    --o-visualization "$num_unadjusted_barplot"


Saved Visualization to: ../data/processed/19_differential_abundance_BMI/numeric_BMI/unadjusted_num_ancombc2/tabulate_BMI_num_unadjusted.qzv
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/numeric_BMI/unadjusted_num_ancombc2/barplot_BMI_num_unadjusted.qzv


In [14]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(num_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

BMI_num = "bmi_sample"

results = pd.DataFrame({
    "log_fold_change_per_year": tables["lfc"][BMI_num],
    "q_value": tables["q"][BMI_num],
    "diff_robust": tables["diff_robust"][BMI_num],
})

significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")

print(f"Unadjusted continuous BMI model — {len(significant)} taxa with q < 0.05")
display(significant)

Unadjusted continuous BMI model — 5 taxa with q < 0.05


,log_fold_change_per_year,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,-0.355699,2.000000e-10,False
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Cladosporiales;f__Cladosporiaceae;g__Cladosporium,-0.152885,4.476294e-04,False
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,-0.181857,4.923228e-04,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Debaryomycetaceae;g__Debaryomyces,0.150578,3.459102e-03,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Kluyveromyces,0.137966,1.256638e-02,False


#### Unadjusted results

Five genera had q < 0.05 for their association with continuous BMI. LFC is expressed per 1 kg/m² increase in BMI.

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Clavispora* | −0.3557 | 2.00 × 10⁻¹⁰ | False |
| *Cladosporium* | −0.1529 | 0.000448 | False |
| *Mucor* | −0.1819 | 0.000492 | False |
| *Debaryomyces* | +0.1506 | 0.003459 | False |
| *Kluyveromyces* | +0.1380 | 0.012566 | False |



Participants with higher BMI tended to have lower estimated abundance of *Clavispora*, *Cladosporium*, and *Mucor*, and higher estimated abundance of *Debaryomyces* and *Kluyveromyces*. None of the five BMI associations passed the sensitivity check for zero counts, so we interpret these patterns cautiously.

### 3.2 Adjusted ANCOM-BC2

We observe age-related shifts in mycobial abundance after accounting for sex and settlement. We use the same 141 samples as in the unadjusted model.

In [20]:
!qiime composition ancombc2 \
    --i-table "$BMI_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + sex_sample + diet_type_sample + bmi_sample" \
    --p-diff-robust \
    --o-ancombc2-output "$num_adjusted_qza"

!qiime composition tabulate \
    --i-data "$num_adjusted_qza" \
    --o-visualization "$num_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$num_adjusted_qza" \
    --o-visualization "$num_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/19_differential_abundance_BMI/numeric_BMI/ancombc2_BMI_num_adjusted.qza
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/numeric_BMI/adjusted_num_ancombc2/tabulate_BMI_num_adjusted.qzv
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/numeric_BMI/adjusted_num_ancombc2/barplot_BMI_num_adjusted.qzv


In [21]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(num_adjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

BMI_num = "bmi_sample"

results = pd.DataFrame({
    "log_fold_change_per_year": tables["lfc"][BMI_num],
    "q_value": tables["q"][BMI_num],
    "diff_robust": tables["diff_robust"][BMI_num],
})

significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")

print(f"Adjusted continuous BMI model — {len(significant)} taxa with q < 0.05")
display(significant)

Adjusted continuous BMI model — 7 taxa with q < 0.05


,log_fold_change_per_year,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,-0.309602,2.000000e-10,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Filobasidiales;f__Filobasidiaceae;g__Naganishia,-0.502547,3.790000e-08,False
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Pleosporales;f__Sporormiaceae;g__Sporormiella,-0.396640,1.140100e-06,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-0.152284,9.684120e-05,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Debaryomycetaceae;g__Debaryomyces,0.148199,6.762541e-04,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma,0.144251,1.525624e-03,False
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Cladosporiales;f__Cladosporiaceae;g__Cladosporium,-0.106322,2.228897e-03,False


#### Adjusted results

After adjustment for age, diet, and sex, seven taxa had q < 0.05 for their association with continuous BMI. LFC is expressed per 1 kg/m² increase in BMI.

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Clavispora* | −0.3096 | 2.00 × 10⁻¹⁰ | False |
| *Naganishia* | −0.5025 | 3.79 × 10⁻⁸ | False |
| *Sporormiella* | −0.3966 | 1.14 × 10⁻⁶ | False |
| Dipodascaceae\* | −0.1523 | 0.000097 | False |
| *Debaryomyces* | +0.1482 | 0.000676 | False |
| *Oligophagozyma* | +0.1443 | 0.001526 | False |
| *Cladosporium* | −0.1063 | 0.002229 | False |

\*Genus unidentified.


In both models, participants with higher BMI tended to have lower estimated abundance of *Clavispora* and *Cladosporium* and higher estimated abundance of *Debaryomyces*. Four additional taxa reached q < 0.05 only after adjustment. Of these, *Naganishia*, *Sporormiella*, and the unidentified Dipodascaceae taxon had lower estimated abundance at higher BMI, while *Oligophagozyma* had higher estimated abundance. None of the seven adjusted associations passed the sensitivity check for zero counts, so we interpret these patterns cautiously.

## 4. BMI as a categorical predictor

We use the collapsed BMI values into the labels "Underweight", "Healthy weight", "Overweight", and "Obese". "Healthy weight" serves as the reference to which we compare mycobial abundance of the other categories. 

### 4.1 Unadjusted ANCOM-BC2

In [7]:
!qiime composition ancombc2 \
    --i-table "$BMI_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "bmi_group" \
    --p-reference-levels "bmi_group::Healthy weight" \
    --p-diff-robust \
    --o-ancombc2-output "$cat_unadjusted_qza"

!qiime composition tabulate \
    --i-data "$cat_unadjusted_qza" \
    --o-visualization "$cat_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$cat_unadjusted_qza" \
    --o-visualization "$cat_unadjusted_barplot"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/19_differential_abundance_BMI/categorical_BMI/ancombc2_BMI_cat_unadjusted.qza
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/categorical_BMI/unadjusted_cat_ancombc2/tabulate_BMI_cat_unadjusted.qzv
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/categorical_BMI/unadjusted_cat_ancombc2/barplot_BMI_cat_unadjusted.qzv


In [5]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(cat_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["Underweight", "Overweight", "Obese"]:
    contrast = f"bmi_group::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. Healthy weight — {len(significant)} taxa with q < 0.05")
    display(significant)

Underweight vs. Healthy weight — 2 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-5.038727,0.000032,False
k__Fungi;p__Basidiomycota;c__Microbotryomycetes;o__Sporidiobolales;f__Sporidiobolaceae;g__Rhodotorula,3.695939,0.016773,False


Overweight vs. Healthy weight — 4 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,-2.723347,0.000022,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Kluyveromyces,3.039913,0.000315,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Saccharomyces,-1.770734,0.011019,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-1.655333,0.035121,False


Obese vs. Healthy weight — 2 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,-3.878842,0.000783,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Debaryomycetaceae;g__Debaryomyces,3.378347,0.020705,False


#### Unadjusted results

Healthy weight was the reference category for all three comparisons.



**Underweight vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae\* | −5.039 | 0.000032 | False |
| *Rhodotorula* | +3.696 | 0.016773 | False |



**Overweight vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Clavispora* | −2.723 | 0.000022 | False |
| *Kluyveromyces* | +3.040 | 0.000315 | False |
| *Saccharomyces* | −1.771 | 0.011019 | False |
| Dipodascaceae\* | −1.655 | 0.035121 | False |



**Obese vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Mucor* | −3.879 | 0.000783 | False |
| *Debaryomyces* | +3.378 | 0.020705 | False |

\*Genus unidentified.



Seven distinct taxa had q < 0.05 across the three comparisons. The unidentified Dipodascaceae taxon had lower estimated abundance in both the underweight and overweight groups than in the healthy weight group. The overweight group also had lower estimated abundance of *Clavispora* and higher estimated abundance of *Kluyveromyces*, while the obese group had lower estimated abundance of *Mucor* and higher estimated abundance of *Debaryomyces*. These four genera showed the same directions in the unadjusted continuous BMI model. None of the eight categorical associations passed the sensitivity check for zero counts, so we interpret these patterns cautiously.

### 4.2 Adjusted ANCOM-BC2

In [6]:
!qiime composition ancombc2 \
    --i-table "$BMI_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + sex_sample + diet_type_sample + bmi_group" \
    --p-reference-levels "bmi_group::Healthy weight" \
    --p-diff-robust \
    --o-ancombc2-output "$cat_adjusted_qza"

!qiime composition tabulate \
    --i-data "$cat_adjusted_qza" \
    --o-visualization "$cat_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$cat_adjusted_qza" \
    --o-visualization "$cat_adjusted_barplot"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/19_differential_abundance_BMI/categorical_BMI/ancombc2_BMI_cat_adjusted.qza
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/categorical_BMI/adjusted_cat_ancombc2/tabulate_BMI_cat_adjusted.qzv
Saved Visualization to: ../data/processed/19_differential_abundance_BMI/categorical_BMI/adjusted_cat_ancombc2/barplot_BMI_cat_adjusted.qzv


#### Adjusted results

After adjustment for age, diet, and sex, five taxa had q < 0.05 across the three comparisons. All five had lower estimated abundance in the respective BMI group than in the healthy weight reference group.



**Underweight vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae\* | −4.285 | 0.000407 | False |
| *Oligophagozyma* | −4.183 | 0.011938 | False |



**Overweight vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Clavispora* | −1.989 | 0.021525 | False |
| *Sporormiella* | −2.290 | 0.044925 | False |



**Obese vs. Healthy weight**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Mucor* | −3.844 | 0.014847 | False |

\*Genus unidentified.



Three associations had q < 0.05 in both models. The unidentified Dipodascaceae taxon had lower estimated abundance in the underweight group, *Clavispora* in the overweight group, and *Mucor* in the obese group. *Oligophagozyma* and *Sporormiella* reached q < 0.05 only after adjustment. None of the five adjusted associations passed the sensitivity check for zero counts, so we interpret these patterns cautiously.

## 4. Discussion

*Clavispora* showed a consistent pattern across both analyses. Its estimated abundance was lower at higher BMI and lower in the overweight group than in the healthy weight group, both before and after adjustment. Other findings depended on how we modeled BMI. For example, *Mucor* had lower estimated abundance in the obese group in both models, but its association with continuous BMI reached q < 0.05 only before adjustment. None of the associations passed the sensitivity check for zero counts, so the evidence for BMI-related differences remains limited.